# ChroniCare: Reinforcement Learning Recommendation Engine
This notebook describes the discrete **Q-Learning agent** which serves as ChroniCare's personalized treatment and lifestyle advisor.

### RL Framing:
- **State space**: Discretized indices of Glucose (0: Normal, 1: Prediabetic, 2: Diabetic), BMI (0: Normal, 1: Overweight, 2: Obese), and Blood Pressure (0: Normal, 1: Prehypertension, 2: Hypertension). Total states = 27.
- **Action space**: 4 lifestyle recommendations:
  - Action 0: Carb Reduction
  - Action 1: Adjust Exercise
  - Action 2: Monitor Vitals
  - Action 3: Professional Consultation
- **Reward function**: Modeled clinically based on lifestyle actions moving patients toward healthier vital thresholds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n_states = 27
n_actions = 4
q_table = np.zeros((n_states, n_actions))

def encode_state(g, bmi, bp):
    return g * 9 + bmi * 3 + bp

def decode_state(state):
    g = state // 9
    bmi = (state % 9) // 3
    bp = state % 3
    return g, bmi, bp

## Q-Learning Training Algorithm

In [ ]:
alpha = 0.1
gamma = 0.9
epsilon = 0.3
episodes = 5000

for ep in range(episodes):
    # Random starting state
    g, bmi, bp = np.random.choice([0,1,2]), np.random.choice([0,1,2]), np.random.choice([0,1,2])
    state = encode_state(g, bmi, bp)
    done = False
    steps = 0
    
    while not done and steps < 10:
        # Action selection (epsilon-greedy)
        if np.random.rand() < epsilon:
            action = np.random.choice(n_actions)
        else:
            action = np.argmax(q_table[state])
            
        next_g, next_bmi, next_bp = g, bmi, bp
        reward = 0
        
        if action == 0:
            if g > 0:
                next_g = max(0, g - 1)
                reward += 15
            else: reward += 5
        elif action == 1:
            if bmi > 0:
                next_bmi = max(0, bmi - 1)
                reward += 15
            if bp > 0:
                next_bp = max(0, bp - 1)
                reward += 10
            else: reward += 5
        elif action == 2:
            reward += 8
        elif action == 3:
            if g == 2 or bmi == 2 or bp == 2:
                next_g = max(0, g - 1)
                next_bp = max(0, bp - 1)
                reward += 20
            else: reward += 5
            
        reward -= (next_g * 6 + next_bmi * 4 + next_bp * 5)
        next_state = encode_state(next_g, next_bmi, next_bp)
        
        # Q-learning Bellman Update
        q_table[state, action] = q_table[state, action] + alpha * (
            reward + gamma * np.max(q_table[next_state]) - q_table[state, action]
        )
        
        g, bmi, bp = next_g, next_bmi, next_bp
        state = next_state
        steps += 1
        if g == 0 and bmi == 0 and bp == 0:
            done = True

print("Learned Q-table (States x Actions):")
print(q_table[:5])